In [ ]:
# セル0: GPU確認
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')

In [ ]:
# セル1: Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# セル2: リポジトリ clone + 作業ディレクトリ移動
import os

REPO_URL = 'https://github.com/stangler/jp-natural-voice-app'
REPO_DIR = '/content/jp-natural-voice-app'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'既にクローン済み: {REPO_DIR}')
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'作業ディレクトリ: {os.getcwd()}')

In [ ]:
# セル3: pip install
# ResolutionTooDeep が出ても末尾 ✅ を確認して先へ進む
import os
os.chdir('/content/jp-natural-voice-app')

# torch 2.4.0+cu124 を固定インストール（バージョン競合回避）
!pip install torch==2.4.0+cu124 torchaudio==2.4.0+cu124 \
    --index-url https://download.pytorch.org/whl/cu124 -q

# Colab 用 requirements
!pip install -r requirements-colab.txt -q 2>&1 | tail -5

# requirements.txt に含まれない可能性がある依存パッケージを個別インストール
!pip install pyopenjtalk loguru onnxruntime pyloudnorm -q
!pip install huggingface_hub==0.24.0 -q

# safetensors（pretrained/jp 変換用）
!pip install safetensors -q

print('✅ pip install 完了')

In [ ]:
# セル3.5: jax パッチ（★新セッションごとに必須）
# transformers が jax を import しようとして壊れる問題を永続パッチで回避
import re, pathlib

target = pathlib.Path('/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py')
if not target.exists():
    # Python バージョンが異なる場合のフォールバック
    import glob
    candidates = glob.glob('/usr/local/lib/python3*/dist-packages/transformers/utils/generic.py')
    if candidates:
        target = pathlib.Path(candidates[0])
    else:
        print('⚠️  generic.py が見つかりません。transformers のインストールを確認してください。')

if target.exists():
    src = target.read_text()
    # 'import jax.numpy as jnp' を pass に置換
    patched = re.sub(
        r'^([ \t]*)(import jax\.numpy as jnp)',
        r'\1pass  # patched_jax: \2',
        src,
        flags=re.MULTILINE
    )
    if patched != src:
        target.write_text(patched)
        print(f'✅ jax パッチ適用完了: {target}')
    else:
        print('✅ すでにパッチ済み（またはパターン不一致）')

# 動作確認
import importlib
try:
    import torch
    from transformers import AutoTokenizer
    print(f'✅ torch={torch.__version__}, transformers import OK')
except Exception as e:
    print(f'❌ import エラー: {e}')

In [ ]:
# セル4: bert_feature.py パッチ（word2ph/text 長さ不一致 AssertionError 回避）
import pathlib
import os
os.chdir('/content/jp-natural-voice-app')

target = pathlib.Path('style_bert_vits2/nlp/japanese/bert_feature.py')
if target.exists():
    src = target.read_text()
    old = 'assert len(word2ph) == len(text)'
    new = '# assert len(word2ph) == len(text)  # patched: skip assertion'
    if old in src:
        target.write_text(src.replace(old, new))
        print('✅ bert_feature.py パッチ適用完了')
    else:
        print('✅ Pattern not found / already patched')
else:
    print(f'⚠️  ファイルが見つかりません: {target}')

In [ ]:
# セル5: lightning_fabric パッチ + HuggingFace ログイン + pretrained モデル配置
import os
os.chdir('/content/jp-natural-voice-app')

# --- lightning_fabric 互換パッチ ---
import pathlib, re
for p in pathlib.Path('/usr/local/lib').rglob('lightning_fabric/utilities/seed.py'):
    src = p.read_text()
    patched = src.replace(
        'from torch import Tensor',
        'try:\n    from torch import Tensor\nexcept ImportError:\n    pass  # patched'
    )
    if patched != src:
        p.write_text(patched)
        print(f'✅ lightning_fabric パッチ適用: {p}')
    else:
        print('✅ lightning_fabric パッチ: skip（不要 or 適用済み）')
    break
else:
    print('✅ lightning_fabric パッチ: skip（ファイル不在）')

# --- HuggingFace ログイン ---
# 初回セッションのみ必要。read token を貼り付けてください。
# !huggingface-cli login
# ↑ 必要な場合はコメントアウトを外して実行

# --- pretrained BERT モデル配置 ---
# Drive に保存済みの場合はそこからコピー（高速・HuggingFace 不要）
DRIVE_PRETRAINED = '/content/drive/MyDrive/StyleBertVITS2/pretrained'

def setup_pretrained(src_name, dst_dir, hf_repo):
    if os.path.exists(dst_dir) and os.listdir(dst_dir):
        print(f'✅ {dst_dir} は配置済み（スキップ）')
        return
    os.makedirs(dst_dir, exist_ok=True)
    drive_src = f'{DRIVE_PRETRAINED}/{src_name}'
    if os.path.exists(drive_src):
        print(f'Drive から {src_name} をコピー中...')
        !cp -r {drive_src}/* {dst_dir}/
        print(f'✅ {dst_dir} コピー完了')
    else:
        print(f'HuggingFace から {hf_repo} を clone 中（数分かかります）...')
        !GIT_LFS_SKIP_SMUDGE=1 git clone https://huggingface.co/{hf_repo} {dst_dir}
        !cd {dst_dir} && git lfs pull
        print(f'✅ {dst_dir} clone 完了')

setup_pretrained('jp',  'pretrained/jp',  'cl-tohoku/bert-base-japanese-v3')
setup_pretrained('en',  'pretrained/en',  'bert-base-uncased')
setup_pretrained('zh',  'pretrained/zh',  'bert-base-chinese')

# pretrained/jp の中身確認
print('\n=== pretrained/jp の中身 ===')
!ls pretrained/jp/ 2>/dev/null || echo '❌ pretrained/jp が空です'

# safetensors のみの場合 pytorch_model.bin に変換
if (not os.path.exists('pretrained/jp/pytorch_model.bin')
        and os.path.exists('pretrained/jp/model.safetensors')):
    print('safetensors → pytorch_model.bin に変換します...')
    from safetensors.torch import load_file
    import torch
    state_dict = load_file('pretrained/jp/model.safetensors')
    torch.save(state_dict, 'pretrained/jp/pytorch_model.bin')
    print('✅ 変換完了')
else:
    print('✅ pytorch_model.bin は存在 or 変換不要')

In [ ]:
# セル6: Drive からデータコピー（raw/ に直接配置）★修正版
import os
os.chdir('/content/jp-natural-voice-app')

DRIVE_WAV_PATH = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample'
wav_count = !find {DRIVE_WAV_PATH}/wavs -name '*.wav' 2>/dev/null | wc -l
esd_exists = os.path.exists(f'{DRIVE_WAV_PATH}/esd.list')
print(f'Drive 音声ファイル数: {wav_count[0]}')
print(f'esd.list: {esd_exists}')

# resample.py の入力は raw/、出力は wavs/
os.makedirs('Data/hirakawa_sample/raw', exist_ok=True)
os.makedirs('Data/hirakawa_sample/wavs', exist_ok=True)

!cp -n {DRIVE_WAV_PATH}/wavs/*.wav Data/hirakawa_sample/raw/
!cp -n {DRIVE_WAV_PATH}/esd.list Data/hirakawa_sample/esd.list 2>/dev/null || true

raw_count = !find Data/hirakawa_sample/raw -name '*.wav' | wc -l
print(f'コピー済み wav (raw/): {raw_count[0]} ファイル ✅')

# Drive に前処理済みファイルがある場合はコピー（2回目以降のセッションでセル8をスキップ）
DRIVE_BERT_PATH = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample_preprocessed'
if os.path.exists(DRIVE_BERT_PATH):
    print('前処理済みファイルを Drive からコピーします...')
    !cp {DRIVE_BERT_PATH}/*.bert.pt Data/hirakawa_sample/wavs/ 2>/dev/null || true
    !cp {DRIVE_BERT_PATH}/*.npy Data/hirakawa_sample/wavs/ 2>/dev/null || true
    !cp {DRIVE_BERT_PATH}/train.list Data/hirakawa_sample/ 2>/dev/null || true
    !cp {DRIVE_BERT_PATH}/val.list Data/hirakawa_sample/ 2>/dev/null || true
    bert_count = !find Data/hirakawa_sample/ -name '*.bert.pt' | wc -l
    npy_count = !find Data/hirakawa_sample/ -name '*.npy' | wc -l
    print(f'.bert.pt: {bert_count[0]}, .npy: {npy_count[0]}')
else:
    print('前処理済みファイルなし → セル7以降で生成します')

In [ ]:
# セル6.5: esd.list パス形式チェック ★新規追加
print('=== esd.list 先頭3行 ===')
!head -3 Data/hirakawa_sample/esd.list

print('\n=== パス形式の自動判定 ===')
with open('Data/hirakawa_sample/esd.list', encoding='utf-8') as f:
    first_line = f.readline().strip()

parts = first_line.split('|')
wav_path = parts[0] if parts else ''
print(f'wav パス例: {wav_path}')

if wav_path.startswith('wavs/'):
    print('✅ 正常: wavs/ 始まり → resample.py の出力先と一致')
elif wav_path.startswith('Data/'):
    print('⚠️  フルパス始まり → preprocess_all.py が正規化するはず（要注意）')
elif wav_path.startswith('raw/'):
    print('❌ 要修正: raw/ 始まり → wavs/ に自動修正します')
    !sed -i 's|^raw/|wavs/|g' Data/hirakawa_sample/esd.list
    print('修正完了 ✅')
    print('修正後の先頭3行:')
    !head -3 Data/hirakawa_sample/esd.list
else:
    print(f'⚠️  想定外のパス形式: {wav_path}')

In [ ]:
# セル7: preprocess_all.py ★ --use_jp_extra を追加（重要）
import os
os.chdir('/content/jp-natural-voice-app')

# 実行前に raw/ の件数を確認
raw_count = !find Data/hirakawa_sample/raw -name '*.wav' | wc -l
print(f'raw/ 件数: {raw_count[0]} ファイル')
assert int(raw_count[0].strip()) > 0, '❌ raw/ にファイルがありません。セル6を先に実行してください。'

!python preprocess_all.py \
    --model_name hirakawa_sample \
    --batch_size 4 \
    --epochs 100 \
    --save_every_steps 200 \
    --use_jp_extra \
    --yomi_error skip

print('\n✅ preprocess_all.py 完了')

In [ ]:
# セル7.7: config.json を学習パラメータで上書き
import json, os
os.chdir('/content/jp-natural-voice-app')

cfg_path = 'Data/hirakawa_sample/config.json'
assert os.path.exists(cfg_path), f'❌ {cfg_path} が存在しません。セル7を先に実行してください。'

with open(cfg_path, encoding='utf-8') as f:
    cfg = json.load(f)

cfg['train']['batch_size'] = 4
cfg['train']['fp16_run'] = True
cfg['train']['eval_interval'] = 200

with open(cfg_path, 'w', encoding='utf-8') as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

print('✅ config.json 更新完了')
print(f'  batch_size    : {cfg["train"]["batch_size"]}')
print(f'  fp16_run      : {cfg["train"]["fp16_run"]}')
print(f'  eval_interval : {cfg["train"]["eval_interval"]}')

In [ ]:
# セル7.8: preprocess_all.py の出力を検証 ★新規追加
import os, json
os.chdir('/content/jp-natural-voice-app')

base = 'Data/hirakawa_sample'

# 1. train.list / val.list の存在と件数確認
for fname in ['train.list', 'val.list']:
    fpath = f'{base}/{fname}'
    if os.path.exists(fpath):
        with open(fpath, encoding='utf-8') as f:
            lines = f.readlines()
        print(f'✅ {fname}: {len(lines)} 件')
        print(f'   先頭1行: {lines[0].strip()}')
    else:
        print(f'❌ {fname} が存在しません')

# 2. config.json の主要パラメータ確認
cfg_path = f'{base}/config.json'
if os.path.exists(cfg_path):
    with open(cfg_path, encoding='utf-8') as f:
        cfg = json.load(f)
    train_cfg = cfg.get('train', {})
    data_cfg = cfg.get('data', {})
    print(f'\n✅ config.json:')
    print(f'   batch_size      : {train_cfg.get("batch_size")}')
    print(f'   fp16_run        : {train_cfg.get("fp16_run")}')
    print(f'   eval_interval   : {train_cfg.get("eval_interval")}')
    print(f'   training_files  : {data_cfg.get("training_files")}')
    print(f'   validation_files: {data_cfg.get("validation_files")}')
else:
    print('❌ config.json が存在しません')

# 3. wavs/ の件数確認（resample の出力）
wav_out = !find {base}/wavs -name '*.wav' | wc -l
print(f'\n✅ wavs/ (44100Hz 変換後): {wav_out[0]} ファイル')

In [ ]:
# セル8: bert_gen + style_gen（初回30〜60分）
# Drive に .bert.pt / .npy がある場合はセル6でコピー済みのためスキップされる
import os
os.chdir('/content/jp-natural-voice-app')

# bert_gen の前に既存ファイル数を確認（スキップ判定）
existing_bert = !find Data/hirakawa_sample/wavs -name '*.bert.pt' | wc -l
print(f'既存 .bert.pt: {existing_bert[0]} ファイル')

if int(existing_bert[0].strip()) == 0:
    print('=== bert_gen 開始 ===')
    !python bert_gen.py -c Data/hirakawa_sample/config.json
    print('✅ bert_gen 完了')
else:
    print('✅ .bert.pt が存在するため bert_gen をスキップ')

# style_gen の前に torchvision をアンインストール（競合回避）
print('\n=== style_gen 前準備（torchvision アンインストール）===')
!pip uninstall torchvision -y -q 2>/dev/null || true

existing_npy = !find Data/hirakawa_sample/wavs -name '*.npy' | wc -l
print(f'既存 .npy: {existing_npy[0]} ファイル')

if int(existing_npy[0].strip()) == 0:
    print('=== style_gen 開始 ===')
    !python style_gen.py --model_name hirakawa_sample
    print('✅ style_gen 完了')
else:
    print('✅ .npy が存在するため style_gen をスキップ')

# 最終件数確認
bert_count = !find Data/hirakawa_sample/wavs -name '*.bert.pt' | wc -l
npy_count = !find Data/hirakawa_sample/wavs -name '*.npy' | wc -l
print(f'\n.bert.pt: {bert_count[0]}, .npy: {npy_count[0]}')

In [ ]:
# セル8.5: 前処理済みファイルを Drive に退避（次回セル8スキップ用）★新規追加
import os
BACKUP = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample_preprocessed'
os.makedirs(BACKUP, exist_ok=True)
!cp Data/hirakawa_sample/wavs/*.bert.pt {BACKUP}/ 2>/dev/null || true
!cp Data/hirakawa_sample/wavs/*.npy {BACKUP}/ 2>/dev/null || true
!cp Data/hirakawa_sample/train.list {BACKUP}/ 2>/dev/null || true
!cp Data/hirakawa_sample/val.list {BACKUP}/ 2>/dev/null || true
backed = !ls {BACKUP} | wc -l
print(f'Drive バックアップ完了: {backed[0]} ファイル ✅')

In [ ]:
# セル9: 学習実行
import os
os.chdir('/content/jp-natural-voice-app')

# 事前チェック
import json
cfg_path = 'Data/hirakawa_sample/config.json'
assert os.path.exists(cfg_path), '❌ config.json が見つかりません'
with open(cfg_path) as f:
    cfg = json.load(f)
print(f'batch_size: {cfg["train"]["batch_size"]}')
print(f'fp16_run:   {cfg["train"]["fp16_run"]}')
print(f'eval_interval: {cfg["train"]["eval_interval"]}')

bert_count = !find Data/hirakawa_sample/wavs -name '*.bert.pt' | wc -l
print(f'.bert.pt: {bert_count[0]} ファイル')
assert int(bert_count[0].strip()) > 0, '❌ .bert.pt が存在しません。セル8を先に実行してください。'

print('\n=== 学習開始 ===')
!python train_ms_jp_extra.py \
    -c Data/hirakawa_sample/config.json \
    -m hirakawa_sample

In [ ]:
# セル10: 学習済みモデルを Drive にバックアップ
import os, glob

BACKUP_MODEL = '/content/drive/MyDrive/StyleBertVITS2/hirakawa_sample_model'
os.makedirs(BACKUP_MODEL, exist_ok=True)

# モデルファイルの場所を確認
model_files = glob.glob('Data/hirakawa_sample/G_*.pth') + \
              glob.glob('Data/hirakawa_sample/D_*.pth') + \
              glob.glob('Data/hirakawa_sample/*.safetensors')
print(f'学習済みモデルファイル: {len(model_files)} 個')
for f in sorted(model_files):
    print(f'  {f}')

# config.json もバックアップ
!cp Data/hirakawa_sample/config.json {BACKUP_MODEL}/ 2>/dev/null || true

# モデルファイルをバックアップ
!cp Data/hirakawa_sample/G_*.pth {BACKUP_MODEL}/ 2>/dev/null || true
!cp Data/hirakawa_sample/D_*.pth {BACKUP_MODEL}/ 2>/dev/null || true
!cp Data/hirakawa_sample/*.safetensors {BACKUP_MODEL}/ 2>/dev/null || true

backed = !ls {BACKUP_MODEL} | wc -l
print(f'\n✅ Drive モデルバックアップ完了: {backed[0]} ファイル → {BACKUP_MODEL}')

In [ ]:
# セル11: 推論テスト
import os, glob
os.chdir('/content/jp-natural-voice-app')

# 最新の Generator モデルを取得
g_models = sorted(glob.glob('Data/hirakawa_sample/G_*.pth'))
if not g_models:
    print('❌ 学習済みモデルが見つかりません。セル9の学習が完了しているか確認してください。')
else:
    latest_model = g_models[-1]
    print(f'使用モデル: {latest_model}')

    test_text = 'こんにちは、テストです。音声合成がうまくいきますように。'
    output_wav = 'test_output.wav'

    !python infer.py \
        --model_file {latest_model} \
        --config_file Data/hirakawa_sample/config.json \
        --text "{test_text}" \
        --output {output_wav} \
        --language JP

    if os.path.exists(output_wav):
        from IPython.display import Audio, display
        display(Audio(output_wav))
        print(f'✅ 推論完了: {output_wav}')
    else:
        print('⚠️  出力ファイルが生成されませんでした。エラーログを確認してください。')